# Stage 01 — Final Strict-Software Universe

## Read-only thesis reproduction

This notebook reconstructs the accepted **272-firm strict-software universe** from frozen local artifacts. It does not call an LLM, SEC, or any network service; it does not write a run artifact, modify sources, stage, commit, or push.

It is a transparent reconstruction, not a claim that every historical exploratory model call was governed. The V9 product-gate full run is governed; the two short refinement experiments are preserved as a hash-bound historical-prototype import because their original launcher and provider capture logs were not retained.

## What this establishes

The notebook checks the following chain and then reconstructs the final membership set:

1. **Item 1 packet corpus:** 7,042 filing packets.
2. **High-recall screen plus human overlay:** 4,045 candidate filings.
3. **Annual-coverage restriction:** 2,799 firms with a filing in every calendar year 2022–2025.
4. **Governed V9 product gate:** 1,268 software-product candidates.
5. **Two short Item 1 refinements:** 279 firms are jointly `STRICT_CORE` and `CORE`.
6. **Accepted human boundary decisions:** exclude nine firms whose main business is games; include Arteris and CEVA after Item 1 reading.

The last operation is therefore `279 − 9 + 2 = 272`. It is intentionally visible in code rather than hidden inside an opaque model output.

In [ ]:
from __future__ import annotations

import hashlib
import json
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'data').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the repository.')


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]


def git(*args: str) -> str:
    return subprocess.check_output(['git', *args], cwd=REPO_ROOT, text=True).strip()


REPO_ROOT = find_repo_root()
print(f'Repository: {REPO_ROOT}')
print('This notebook is read-only: no model, SEC, or network call is made.')

## 1. Repository identity and safe mode

A dirty worktree is reported, never repaired. The repository revision documents the code and prompt files used to verify the persisted artifacts; the artifacts themselves are not rewritten here.

In [ ]:
identity = {
    'branch': git('branch', '--show-current'),
    'head': git('rev-parse', 'HEAD'),
    'worktree_status': git('status', '--short', '-uall') or 'clean',
}
display(pd.DataFrame([identity]).T.rename(columns={0: 'value'}))

## 2. Locate the frozen inputs

Every path below is an existing local artifact. The constants make the exact thesis snapshot visible and prevent a later similarly named run from being selected implicitly.

In [ ]:
PACKET_RUN = REPO_ROOT / 'data/runs/baseline-packets/baseline-packets-domestic-text-lineage-v5-20260819'
PACKET_MANIFEST = PACKET_RUN / 'baseline_packet_manifest.json'
SCREEN_RELEASE = REPO_ROOT / 'data/runs/universe-screen-releases/universe-screen-release-v1-20260823/universe_screen_release_manifest.json'
CANDIDATE_COHORT = REPO_ROOT / 'data/runs/universe-classifier-candidate-cohorts/universe-classifier-candidate-cohort-v1-20260824/universe_classifier_candidate_cohort_manifest.json'
COVERAGE_RUN = REPO_ROOT / 'data/runs/universe-annual-coverage-cohorts/universe-annual-coverage-cohort-v1-20260829'
COVERAGE_MANIFEST = COVERAGE_RUN / 'universe_annual_coverage_cohort_manifest.json'
V9_RUN = REPO_ROOT / 'data/runs/universe-classifier-product-gate-v9-aggregates/universe-classifier-product-gate-v9-aggregate-20260901'
V9_MANIFEST = V9_RUN / 'universe_classifier_product_gate_v9_aggregate_manifest.json'
V9_CANDIDATES = V9_RUN / 'universe_classifier_product_gate_v9_software_candidates.jsonl'
PROTOTYPE_RUN = REPO_ROOT / 'data/runs/universe-final-prototype-imports/universe-final-prototype-import-20260902'
PROTOTYPE_MANIFEST = PROTOTYPE_RUN / 'universe_final_software_prototype_import_manifest.json'
GAME_DRAFT = REPO_ROOT / 'data/runs/universe-final-strict-software-adjudication-drafts/universe-final-strict-software-adjudication-draft-20260902/agreed_core_game_postfilter_candidates.jsonl'

for path in (PACKET_MANIFEST, SCREEN_RELEASE, CANDIDATE_COHORT, COVERAGE_MANIFEST, V9_MANIFEST, V9_CANDIDATES, PROTOTYPE_MANIFEST, GAME_DRAFT):
    assert path.is_file(), f'Missing frozen input: {path}'
print('All frozen inputs are present.')

## 3. Reproduce the corpus, screen, and annual-coverage counts

The high-recall result is a broad starting population, not the strict-software decision. The annual filter is applied **after** that screen and asks only whether the filing panel is usable for the 2022–2025 research window.

In [ ]:
packet = load_json(PACKET_MANIFEST)
release = load_json(SCREEN_RELEASE)
cohort = load_json(CANDIDATE_COHORT)
coverage = load_json(COVERAGE_MANIFEST)

for filename, expected_hash in packet['output_hashes'].items():
    assert sha256_file(PACKET_RUN / filename) == expected_hash, filename
release_records = SCREEN_RELEASE.parent / 'universe_screen_release_records.jsonl'
cohort_records = CANDIDATE_COHORT.parent / 'universe_classifier_candidate_records.jsonl'
assert sha256_file(release_records) == release['output_hashes']['universe_screen_release_records.jsonl']
assert sha256_file(cohort_records) == cohort['output_hashes']['universe_classifier_candidate_records.jsonl']
assert cohort['sources']['release']['manifest_sha256'] == sha256_file(SCREEN_RELEASE)
assert cohort['sources']['release']['records_jsonl_sha256'] == sha256_file(release_records)
assert sha256_file(COVERAGE_RUN / 'universe_annual_coverage_cohort_records.jsonl') == coverage['output_hashes']['universe_annual_coverage_cohort_records.jsonl']
assert sha256_file(COVERAGE_RUN / 'universe_annual_coverage_cohort_exclusions.jsonl') == coverage['output_hashes']['universe_annual_coverage_cohort_exclusions.jsonl']
assert coverage['sources']['candidate_cohort']['manifest_sha256'] == sha256_file(CANDIDATE_COHORT)
assert coverage['sources']['candidate_cohort']['records_jsonl_sha256'] == sha256_file(cohort_records)
assert coverage['counts']['included'] + coverage['counts']['excluded'] == coverage['counts']['source_cohort_rows']

stage_counts = pd.DataFrame([
    {'stage': 'Item 1 packets', 'firms_or_filings': packet['counts']['packets_built'], 'meaning': 'frozen Item 1 evidence packets'},
    {'stage': 'high-recall candidate cohort', 'firms_or_filings': coverage['counts']['source_cohort_rows'], 'meaning': 'model screen plus human overlay'},
    {'stage': 'annual-coverage cohort', 'firms_or_filings': coverage['counts']['included'], 'meaning': '2022–2025 filing continuity'},
])
display(stage_counts)
display(pd.DataFrame([coverage['counts']]).T.rename(columns={0: 'annual_coverage_count'}))

## 4. Verify the governed V9 product gate

V9 used `software_universe_classifier_pilot.v9.md` over the annual-coverage cohort. Its `YES` rows are candidates for stricter review; they are not, by themselves, final membership.

In [ ]:
v9 = load_json(V9_MANIFEST)
assert sha256_file(V9_CANDIDATES) == v9['output_hashes']['universe_classifier_product_gate_v9_software_candidates.jsonl']
assert v9['coverage_cohort_manifest_sha256'] == sha256_file(COVERAGE_MANIFEST)
v9_prompt = REPO_ROOT / v9['prompt_template_path']
assert sha256_file(v9_prompt) == v9['prompt_template_sha256']

v9_summary = pd.DataFrame([v9['counts']]).T.rename(columns={0: 'V9 count'})
display(v9_summary)
print(f'Verified V9 prompt: {v9_prompt.relative_to(REPO_ROOT)}')

## 5. Verify the two short refinement experiments

The strict prompt asks whether customers buy a separately identifiable software product that is central to the firm. The centrality prompt asks whether software itself is the principal customer purchase. Their frozen outputs are imported verbatim and hash-bound to their source candidate file and version-controlled prompt bytes.

Eleven strict-prompt rows carry an explicit historical transport failure; they are not silently turned into `NOT_STRICT_CORE`. The centrality prompt completed all 1,268 rows.

In [ ]:
prototype = load_json(PROTOTYPE_MANIFEST)
assert prototype['no_model_call'] is True
assert prototype['source_candidates']['sha256'] == sha256_file(V9_CANDIDATES)
for filename, expected_hash in prototype['output_hashes'].items():
    assert sha256_file(PROTOTYPE_RUN / filename) == expected_hash, filename
for experiment in prototype['experiments'].values():
    prompt_path = REPO_ROOT / experiment['prompt_path']
    assert sha256_file(prompt_path) == experiment['prompt_sha256']

display(pd.DataFrame([prototype['counts']]).T.rename(columns={0: 'count'}))

## 6. Reconstruct the common strict-software set

A filing enters the mechanical base only when both experiments agree: `STRICT_CORE` in the strict prompt and `CORE` in the centrality prompt. This avoids treating either one as ground truth.

In [ ]:
strict_rows = load_jsonl(PROTOTYPE_RUN / 'strict_core_refinement_outputs.jsonl')
centrality_rows = load_jsonl(PROTOTYPE_RUN / 'software_centrality_refinement_outputs.jsonl')
candidate_rows = load_jsonl(V9_CANDIDATES)

key = lambda row: (row['cik'], row['accession'])
strict_core = {key(row) for row in strict_rows if row['status'] == 'completed' and row['model_output']['strict_core'] == 'STRICT_CORE'}
centrality_core = {key(row) for row in centrality_rows if row['status'] == 'completed' and row['model_output']['software_centrality'] == 'CORE'}
common_core = strict_core & centrality_core
candidate_by_key = {key(row): row for row in candidate_rows}

assert len(candidate_by_key) == 1268
assert len(strict_core) == prototype['counts']['strict_core_rows'] == 358
assert len(centrality_core) == prototype['counts']['centrality_core_rows'] == 334
assert len(common_core) == prototype['counts']['intersection_rows'] == 279
print(f'Common strict-software base: {len(common_core)} filings')

## 7. Apply the accepted transparent boundary decisions

These are human research decisions, not model predictions:

- Remove the nine common-core firms whose main business is games.
- Add **Arteris** and **CEVA**. Each was `STRICT_CORE` in the strict prompt but `CO_ESSENTIAL` in the centrality prompt; Item 1 describes software/IP licensed to external customers.

The raw identities of the nine game exclusions come from the frozen adjudication draft. The two additions are stated explicitly here so a reader can audit the exact departure from the intersection rule.

In [ ]:
game_rows = load_jsonl(GAME_DRAFT)
game_exclusions = {key(row) for row in game_rows}
human_inclusions = {
    ('0001667011', '0001667011-22-000010'): {'issuer_name': 'Arteris, Inc.', 'reason': 'software/IP licensed to external semiconductor customers'},
    ('0001173489', '0001437749-22-004905'): {'issuer_name': 'CEVA INC', 'reason': 'software and IP licensed to external customers'},
}

assert len(game_exclusions) == 9
assert game_exclusions <= common_core
assert set(human_inclusions).isdisjoint(common_core)
assert set(human_inclusions) <= set(candidate_by_key)

final_keys = (common_core - game_exclusions) | set(human_inclusions)
assert len(final_keys) == 272
assert len(final_keys) == len(common_core) - len(game_exclusions) + len(human_inclusions)

decision_accounting = pd.DataFrame([
    {'operation': 'common strict/core intersection', 'change': 279, 'running_total': 279},
    {'operation': 'exclude main-business game firms', 'change': -len(game_exclusions), 'running_total': 279 - len(game_exclusions)},
    {'operation': 'include Arteris and CEVA', 'change': len(human_inclusions), 'running_total': len(final_keys)},
])
display(decision_accounting)

## 8. Final thesis universe

The table is the reproducible final membership list. `membership_basis` distinguishes the 270 mechanical intersection members from the two explicit human inclusions. It is a firm/filing universe for the Item 1-visible research panel; it is not a general claim about every software company or a product–capability–task dataset yet.

In [ ]:
final_rows = []
for cik, accession in sorted(final_keys):
    candidate = candidate_by_key[(cik, accession)]
    inclusion = human_inclusions.get((cik, accession))
    final_rows.append({
        'cik': cik,
        'accession': accession,
        'membership_basis': 'explicit_human_inclusion' if inclusion else 'two_prompt_intersection',
        'human_reason': inclusion['reason'] if inclusion else None,
        'v9_confidence': candidate['axes']['confidence'],
    })
FINAL_UNIVERSE = pd.DataFrame(final_rows)
assert len(FINAL_UNIVERSE) == 272
display(FINAL_UNIVERSE.head(20))
display(FINAL_UNIVERSE['membership_basis'].value_counts().rename_axis('membership_basis').to_frame('firms'))

## 9. Interpretation and limitations

- The evidence scope is annual SEC Item 1 text. It is deliberately stable and reproducible, but does not claim to describe all current product pages or full product history.
- The 2022–2025 filing rule conditions the sample on continuing SEC reporting; this is appropriate for a panel but excludes firms without that continuity.
- The 272 firms are a strict-software research universe, not labels for the other 2,527 annual-coverage firms.
- The V9 model output and historical refinements are inputs to a transparent selection rule; they are not human gold labels.
- The next PCT stage should create annual product/capability/task observations for these firms and preserve `unknown` where Item 1 is silent.

## Session-closing checklist

Before submitting or extending the thesis, record the repository commit, preserve the imported prototype artifact and all upstream run directories, and run this notebook top-to-bottom in a clean environment. Any new membership decision must be added as a separately documented successor; do not edit the frozen historical prompts or outputs in place.